# Setup

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import sys
from pathlib import Path
from hydra.utils import instantiate
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from src.utils.notebook_setup import init_nlp_notebook

# 1. Вычисляем корень проекта ОДИН РАЗ
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Регистрируем все резолверы ГЛОБАЛЬНО
# Теперь ${paths.data_dir} будет подтягиваться из конфига, 
# а если Hydra падает, мы подставляем ${PROJECT_ROOT}
OmegaConf.register_new_resolver("project_root", lambda: str(PROJECT_ROOT))
OmegaConf.register_new_resolver("hydra", lambda path: str(PROJECT_ROOT), replace=True)
OmegaConf.register_new_resolver("now", lambda *args: "now", replace=True)

# 3. Инициализируем Hydra
cfg = init_nlp_notebook()

# 4. Трюк: принудительно подменяем переменную paths, если она не разрешилась
# Это поможет конфигу увидеть, где лежат данные, без правок yaml-файлов
if "paths" not in cfg:
    cfg.paths = OmegaConf.create()
cfg.paths.data_dir = str(PROJECT_ROOT / "data")

NLP Environment ready. Root: c:\fake-news-detection-ml-system


# Chunking Audit

In [3]:
from llama_index.core import Document
from src.jobs.rag_update_job import clean_text

# Берем тестовый сырой текст (или загружаем из cfg.paths.data_dir)
raw_text = """
Анализ данных — это процесс инспекции, очистки и моделирования данных. 
<br>Цель состоит в обнаружении полезной информации! 
Подробнее: https://example.com/data
"""

# Имитируем логику из rag_update_job.py
cleaned_text = clean_text(raw_text)
test_doc = Document(text=cleaned_text)

# Инициализируем индексатор для доступа к сплиттеру (без сохранения)
indexer = instantiate(cfg.rag.indexer)
nodes = indexer.Settings.node_parser.get_nodes_from_documents([test_doc])

print(f"Original Cleaned Text:\n{cleaned_text}\n")
print("-" * 40)
for i, node in enumerate(nodes):
    print(f"Chunk {i+1} (Length: {len(node.text)}):\n{node.text}\n")

INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.
c:\fake-news-detection-ml-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

# Indexing

In [4]:
# Если базы еще нет, создаем её прямо здесь
# indexer.build_and_save_index() будет читать данные из documents_dir
try:
    retriever = instantiate(cfg.rag.retriever)
    print("Векторная БД успешно загружена.")
except Exception as e:
    print(f"БД не найдена. Собираем индекс...\n{e}")
    indexer = instantiate(cfg.rag.indexer)
    indexer.build_and_save_index() # Загрузит сырые файлы и сохранит в FAISS
    retriever = instantiate(cfg.rag.retriever)

c:\fake-news-detection-ml-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

# Sanity Check Retrieval

In [ ]:
test_query = "Что такое RAG и как он работает?"

# Используем твой базовый метод
context = retriever.retrieve_context(test_query)

print(f"QUERY: {test_query}\n")
print(f"RETRIEVED CONTEXT (Top-{retriever.similarity_top_k}):\n")
print(context)

# Score Distribution


In [ ]:
import numpy as np
import seaborn as sns

queries = [
    "Как настроить параметры Llama-index?",
    "Какие существуют методы классификации?",
    "Привет, как дела?" # Запрос "вне домена", чтобы увидеть плохие скоры
]

all_scores = []

for q in queries:
    # Обращаемся напрямую к внутреннему объекту LlamaIndex
    nodes_with_score = retriever.retriever.retrieve(q)
    scores = [node.score for node in nodes_with_score]
    all_scores.extend(scores)
    
    print(f"Query: '{q}'")
    for i, node in enumerate(nodes_with_score):
        print(f"  [{i+1}] Score: {node.score:.4f} | Text: {node.get_content()[:60]}...")
    print("-" * 50)

# Визуализируем распределение скоров
plt.figure(figsize=(8, 4))
sns.histplot(all_scores, bins=15, kde=True)
plt.title("Distribution of Retrieval Scores (Distances)")
plt.xlabel("Score")
plt.ylabel("Count")
plt.show()

# На основе этого графика ты сможешь решить:
# "Если Score < 0.3 (или > 1.5 для L2), значит контекст нерелевантен, и LLM должна отвечать 'Я не знаю'".